In [12]:
import pandas as pd
import numpy as np
import os

# --- 1. 路径配置 ---
input_dir = r"C:\Users\28616\Desktop\Cluster_Matrices"
output_dir = r"C:\Users\28616\Desktop\Filtered_Lists"
if not os.path.exists(output_dir): os.makedirs(output_dir)

results = []

print("🚿 开始执行第一次过滤：中位数 (A+G) >= 10 且 至少3个样本 G > 0 ...")

for c in range(6):
    file_path = os.path.join(input_dir, f"Cluster_{c}_matrix_12xn.csv")
    
    if os.path.exists(file_path):
        # 读取 12 x N 矩阵
        df = pd.read_csv(file_path, index_col=0)
        original_sites = df.columns.tolist()
        num_original = len(original_sites)
        
        # --- 计算筛选指标 ---
        # 1. 提取 G 和 A 的数据块 (每两行为一组，0/2/4...为G, 1/3/5...为A)
        g_data = df.iloc[0::2].values  # 6 x N
        a_data = df.iloc[1::2].values  # 6 x N
        
        # 2. 计算总深度 (Depth = G + A)
        depths = g_data + a_data
        
        # 3. 计算中位数 (median depth)
        medians = np.median(depths, axis=0)
        
        # 4. 统计每个位点有多少个样本 G > 0
        g_presence_count = (g_data > 0).sum(axis=0)
        
        # 5. 复合筛选条件：中位数 >= 10 且 至少 3 个样本有 G
        keep_mask = (medians >= 10) & (g_presence_count >= 3)
        
        # 6. 获取保留的位点名
        filtered_sites = [original_sites[i] for i, keep in enumerate(keep_mask) if keep]
        num_filtered = len(filtered_sites)
        
        # --- 保存结果 ---
        save_name = f"Cluster_{c}_afterfilter1.txt"
        save_path = os.path.join(output_dir, save_name)
        with open(save_path, 'w') as f:
            for site in filtered_sites:
                f.write(f"{site}\n")
        
        # --- 统计对比 ---
        deleted_count = num_original - num_filtered
        del_pct = (deleted_count / num_original * 100) if num_original > 0 else 0
        
        results.append({
            "Cluster": f"Cluster_{c}",
            "原始位点数": num_original,
            "保留位点数": num_filtered,
            "剔除位点数": deleted_count,
            "剔除比例 (%)": round(del_pct, 2)
        })
        print(f"✅ {save_name} 已生成 (保留: {num_filtered}, 剔除: {deleted_count})")

# --- 2. 打印最终对比报告 ---
df_report = pd.DataFrame(results)
print("\n" + "="*60)
print("📊 第一次过滤统计报告")
print("="*60)
print(df_report.to_string(index=False))

# 保存报告
df_report.to_csv(os.path.join(output_dir, "Filter1_Comparison_Report.csv"), index=False)

🚿 开始执行第一次过滤：中位数 (A+G) >= 10 且 至少3个样本 G > 0 ...
✅ Cluster_0_afterfilter1.txt 已生成 (保留: 3729, 剔除: 97071)
✅ Cluster_1_afterfilter1.txt 已生成 (保留: 2962, 剔除: 97838)
✅ Cluster_2_afterfilter1.txt 已生成 (保留: 2042, 剔除: 98758)
✅ Cluster_3_afterfilter1.txt 已生成 (保留: 1997, 剔除: 98803)
✅ Cluster_4_afterfilter1.txt 已生成 (保留: 1552, 剔除: 99248)
✅ Cluster_5_afterfilter1.txt 已生成 (保留: 1047, 剔除: 99753)

📊 第一次过滤统计报告
  Cluster  原始位点数  保留位点数  剔除位点数  剔除比例 (%)
Cluster_0 100800   3729  97071     96.30
Cluster_1 100800   2962  97838     97.06
Cluster_2 100800   2042  98758     97.97
Cluster_3 100800   1997  98803     98.02
Cluster_4 100800   1552  99248     98.46
Cluster_5 100800   1047  99753     98.96


In [14]:
import anndata
import numpy as np
import pandas as pd
import os
import gc

# --- 1. 配置路径 ---
raw_data_dir = r"C:\Users\28616\Desktop\spatialDE_rawdata\ATOI"
list_dir = r"C:\Users\28616\Desktop\Filtered_Lists"
output_dir = r"C:\Users\28616\Desktop\Filtered_Matrices"
os.makedirs(output_dir, exist_ok=True)

sample_ids = ["49", "50", "51", "52", "53", "54"]

print("🚀 开始执行：基于 Filtered_Lists 进行深度重构...")

for c in range(6):
    list_path = os.path.join(list_dir, f"Cluster_{c}_afterfilter1.txt")
    if not os.path.exists(list_path):
        print(f"⚠️ 列表文件缺失: {list_path}，跳过。")
        continue
    
    # 1. 核心修复：在这里初始化该 Cluster 的数据容器
    # 使用字典存储，key 为索引名（如 49_G, 49_A），value 为数据行
    cluster_data_dict = {}
    
    # 读取本次 Cluster 专有的位点列表
    with open(list_path, 'r') as f:
        target_sites = [line.strip() for line in f if line.strip()]
    
    # 2. 遍历样本提取数据
    for s_id in sample_ids:
        h5ad_path = os.path.join(raw_data_dir, f"GSM81924{s_id}_AtoI_tissue.h5ad")
        
        # 显式打开文件
        adata = anndata.read_h5ad(h5ad_path, backed='r')
        
        # 读取对应 Cluster 的文本，放在样本循环内，确保只加载本次 Cluster 的 Bins
        txt_path = os.path.join(r"C:\Users\28616\Desktop\spCLUE_Results", f"{s_id}_cluster{c}.txt")
        with open(txt_path, 'r') as f:
            target_ids = [line.strip() for line in f if line.strip()]
        
        # 切片提取
        sub = adata[adata.obs_names.isin(target_ids)]
        
        # 3. 使用 Series.reindex 强制对齐 (这是解决位点错乱的万能方法)
        # 即使某个位点在样本中没有表达，reindex 也会自动补 0，保证所有样本的列顺序完全一致
        g_vals = pd.Series(np.ravel(sub.X.sum(axis=0)), index=sub.var_names).reindex(target_sites, fill_value=0)
        a_vals = pd.Series(np.ravel(sub.layers['AGcount_A'].sum(axis=0)), index=sub.var_names).reindex(target_sites, fill_value=0)
        
        cluster_data_dict[f"{s_id}_G"] = g_vals.values
        cluster_data_dict[f"{s_id}_A"] = a_vals.values
        
        # 强制关闭连接并清理内存
        adata.file.close()
        del adata
        
    # 4. 构建 DataFrame 并保存
    # index 自动为 ['49_G', '49_A', ..., '54_A']
    df_new = pd.DataFrame(cluster_data_dict, index=target_sites).T
    
    save_path = os.path.join(output_dir, f"Cluster_{c}_new.csv")
    df_new.to_csv(save_path)
    
    # 强制进行垃圾回收
    gc.collect()
    print(f"✅ Cluster {c}: 重构完成 (列数: {df_new.shape[1]})")

print("\n🎉 所有重构任务全部完成！所有矩阵现在已实现 Cluster 间差异独立。")

🚀 开始执行：基于 Filtered_Lists 进行深度重构...
✅ Cluster 0: 重构完成 (列数: 3729)
✅ Cluster 1: 重构完成 (列数: 2962)
✅ Cluster 2: 重构完成 (列数: 2042)
✅ Cluster 3: 重构完成 (列数: 1997)
✅ Cluster 4: 重构完成 (列数: 1552)
✅ Cluster 5: 重构完成 (列数: 1047)

🎉 所有重构任务全部完成！所有矩阵现在已实现 Cluster 间差异独立。
